# 📖 Prose vs Verse Classifier
### Using a lightweight local LLM

This notebook classifies dialogue/text as either **prose** or **verse** using a local LLM.
Three backend options are provided — pick the one that matches your setup:

| Option | Runtime | Best for |
|--------|---------|----------|
| **A** | [Ollama](https://ollama.com) | Easiest setup, runs as a local server |
| **B** | [llama-cpp-python](https://github.com/abetlen/llama-cpp-python) | Direct GGUF model files, no server needed |
| **C** | [Hugging Face Transformers](https://huggingface.co/docs/transformers) | Full HF ecosystem, GPU/CPU |

---
**Recommended lightweight models:**
- `gemma2:2b` / `tinyllama` (Ollama)
- `Qwen2.5-1.5B-Instruct.Q4_K_M.gguf` (llama-cpp)
- `Qwen/Qwen2.5-1.5B-Instruct` (HuggingFace)

## 0 · Install dependencies
Run only the cell matching your chosen backend.

In [ ]:
# Option A – Ollama  (Ollama app must be running: https://ollama.com/download)
%pip install -q ollama

In [ ]:
# Option B – llama-cpp-python
%pip install -q llama-cpp-python

In [ ]:
# Option C – Hugging Face Transformers
%pip install -q transformers torch accelerate

---
## 1 · Classifier core — shared across all backends

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Protocol, runtime_checkable


class Label(str, Enum):
    PROSE = "prose"
    VERSE = "verse"
    UNKNOWN = "unknown"


@dataclass
class ClassificationResult:
    text: str
    label: Label
    raw_response: str

    def __repr__(self):
        preview = self.text[:60].replace("\n", " ")
        return f'[{self.label.value.upper()}] "{preview}…"'


# ── Prompt ──────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a literary classifier. Your only job is to decide
whether a given piece of text is PROSE or VERSE.

Definitions:
- PROSE: ordinary written language without metrical structure. Sentences flow
  in paragraphs, without deliberate line breaks for rhythm or rhyme.
- VERSE: text that uses deliberate line breaks, meter, rhyme schemes, or
  rhythmic patterns (poetry, songs, ballads, verse drama, etc.).

Rules:
1. Reply with exactly one word: either  prose  or  verse  (lowercase).
2. Do NOT add explanations, punctuation, or any other text."""


def build_user_message(text: str) -> str:
    return f"Classify the following text:\n\n{text.strip()}"


def parse_label(raw: str) -> Label:
    """Extract the label from the model's raw reply."""
    clean = raw.strip().lower().split()[0] if raw.strip() else ""
    if "verse" in clean:
        return Label.VERSE
    if "prose" in clean:
        return Label.PROSE
    return Label.UNKNOWN


print("✅ Core classifier loaded.")

---
## 2 · Backend adapters
### Option A — Ollama

In [ ]:
# ── Option A: Ollama ─────────────────────────────────────────────────────────
# Make sure Ollama is running and the model is pulled:
#   ollama pull gemma2:2b

import ollama

OLLAMA_MODEL = "gemma2:2b"   # ← change to any model you have pulled


def classify_ollama(text: str) -> ClassificationResult:
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_message(text)},
        ],
        options={"temperature": 0, "num_predict": 5},
    )
    raw = response["message"]["content"]
    return ClassificationResult(text=text, label=parse_label(raw), raw_response=raw)


# Quick smoke-test
sample = "To be, or not to be, that is the question."
result = classify_ollama(sample)
print(result)

### Option B — llama-cpp-python (GGUF file)

In [ ]:
# ── Option B: llama-cpp-python ───────────────────────────────────────────────
# Download a GGUF model, e.g. from HuggingFace:
#   https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF

from llama_cpp import Llama

GGUF_PATH = "/path/to/your/model.gguf"   # ← update this path

llm_cpp = Llama(
    model_path=GGUF_PATH,
    n_ctx=512,       # small context window is fine for classification
    n_threads=4,     # adjust to your CPU core count
    verbose=False,
)


def classify_llamacpp(text: str) -> ClassificationResult:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{build_user_message(text)}<|end|>\n"
        f"<|assistant|>\n"
    )
    output = llm_cpp(
        prompt,
        max_tokens=5,
        temperature=0,
        stop=["<|end|>", "\n"],
    )
    raw = output["choices"][0]["text"]
    return ClassificationResult(text=text, label=parse_label(raw), raw_response=raw)


sample = "To be, or not to be, that is the question."
result = classify_llamacpp(sample)
print(result)

### Option C — Hugging Face Transformers

In [ ]:
# ── Option C: Hugging Face Transformers ─────────────────────────────────────
# Model is downloaded automatically on first run.

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ← ~3 GB download, CPU-friendly

tokenizer_hf = AutoTokenizer.from_pretrained(HF_MODEL)
model_hf     = AutoModelForCausalLM.from_pretrained(
    HF_MODEL,
    torch_dtype=torch.float32,   # use torch.float16 if you have a GPU
    device_map="auto",
)

pipe_hf = pipeline(
    "text-generation",
    model=model_hf,
    tokenizer=tokenizer_hf,
    max_new_tokens=5,
    do_sample=False,   # greedy = deterministic
)


def classify_hf(text: str) -> ClassificationResult:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(text)},
    ]
    # Use the model's chat template
    prompt_str = tokenizer_hf.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    output = pipe_hf(prompt_str)
    # Strip the prompt from the generated text
    raw = output[0]["generated_text"][len(prompt_str):].strip()
    return ClassificationResult(text=text, label=parse_label(raw), raw_response=raw)


sample = "To be, or not to be, that is the question."
result = classify_hf(sample)
print(result)

---
## 3 · 🔌 Set your active backend
Point `classify` at whichever function you initialised above.

In [ ]:
# ── Pick ONE of the three ────────────────────────────────────────────────────
classify = classify_ollama       # Option A
# classify = classify_llamacpp   # Option B
# classify = classify_hf         # Option C

print(f"Active backend: {classify.__name__}")

---
## 4 · Sample dialogues

In [ ]:
sample_texts = [
    # ── Verse examples ────────────────────────────────────────────────────────
    """
    To be, or not to be, that is the question:
    Whether 'tis nobler in the mind to suffer
    The slings and arrows of outrageous fortune,
    Or to take arms against a sea of troubles
    And by opposing end them.
    """,

    """
    Two roads diverged in a yellow wood,
    And sorry I could not travel both
    And be one traveler, long I stood
    And looked down one as far as I could
    To where it bent in the undergrowth.
    """,

    # ── Prose examples ────────────────────────────────────────────────────────
    """
    It was the best of times, it was the worst of times, it was the age of
    wisdom, it was the age of foolishness, it was the epoch of belief, it was
    the epoch of incredulity, it was the season of Light, it was the season
    of Darkness.
    """,

    """
    She walked into the room and immediately sensed that something was wrong.
    The furniture had been rearranged, the curtains drawn tight against the
    afternoon sun, and a faint smell of cigarette smoke lingered in the air,
    though no one in the house had ever smoked.
    """,

    # ── Edge case: rhythmic prose ─────────────────────────────────────────────
    """
    And God said, Let there be light: and there was light. And God saw the
    light, that it was good: and God divided the light from the darkness.
    """,
]

print(f"{len(sample_texts)} samples ready.")

---
## 5 · Run classification

In [ ]:
results: list[ClassificationResult] = []

for i, text in enumerate(sample_texts, 1):
    print(f"Classifying sample {i}/{len(sample_texts)}…", end=" ", flush=True)
    r = classify(text)
    results.append(r)
    print(f"→ {r.label.value.upper()}  (raw: '{r.raw_response.strip()}')") 

print("\nDone.")

---
## 6 · Results summary

In [ ]:
from collections import Counter

counts = Counter(r.label for r in results)

print("═" * 60)
print(f"  {'LABEL':<10}  {'COUNT':>5}")
print("─" * 60)
for label, count in counts.items():
    print(f"  {label.value:<10}  {count:>5}")
print("═" * 60)

print("\nDetailed results:")
print("─" * 60)
for i, r in enumerate(results, 1):
    preview = r.text.strip()[:70].replace("\n", " ")
    print(f"{i}. [{r.label.value.upper():>7}]  {preview}…")

---
## 7 · Classify your own text

In [ ]:
my_text = """
Paste or write your dialogue / paragraph here.
It can span multiple lines.
"""

result = classify(my_text)
print(f"Label      : {result.label.value.upper()}")
print(f"Raw reply  : {result.raw_response.strip()}")
print(f"Text       : {result.text.strip()[:120]}")

---
## 8 · Batch classification from a list or CSV

In [ ]:
import csv
import io

def classify_batch(texts: list[str]) -> list[ClassificationResult]:
    """Classify a list of strings and return results."""
    return [classify(t) for t in texts]


def results_to_csv(results: list[ClassificationResult]) -> str:
    """Export results to a CSV string."""
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["index", "label", "raw_response", "text"])
    for i, r in enumerate(results, 1):
        writer.writerow([i, r.label.value, r.raw_response.strip(), r.text.strip()])
    return buf.getvalue()


# Example: classify the samples and export
csv_output = results_to_csv(results)
print(csv_output[:500])

# To save:
# with open("classifications.csv", "w") as f:
#     f.write(csv_output)